# DiffusionGemma prose eval - `ar` arm

Downloads weights from public `gs://gemma-data/`. Nothing is uploaded to Kaggle.


In [ ]:
# --- install -------------------------------------------------------------
# Validated end-to-end in a clean venv 2026-10-03. Three landmines, each of
# which would have burned a whole TPU slot:
#   1. PyPI `gemma` 4.0.1 has NO `diffusion` module -> must install from git
#      (git main is 4.1.0 and does have it).
#   2. `tensorflow-cpu` was REMOVED from PyPI, but gemma and kauldron both
#      still pin it -> pip cannot resolve -> install a stub that forwards to
#      `tensorflow`.
#   3. kauldron imports np.float128, removed in numpy 2 -> pin numpy<2.
import pathlib, subprocess, sys, time

def sh(cmd):
    t = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = p.returncode == 0
    print(f"[{'ok ' if ok else 'FAIL'}] {cmd[:72]}  ({{time.time()-t:.0f}}s)", flush=True)
    if not ok:
        print((p.stderr or p.stdout)[-1200:], flush=True)
    return ok

sh(f"{{sys.executable}} -m pip install -q tensorflow")
_d = pathlib.Path("/tmp/tfstub"); _d.mkdir(exist_ok=True)
(_d / "setup.py").write_text(
    "from setuptools import setup\n"
    "setup(name='tensorflow-cpu', version='2.21.0', install_requires=['tensorflow'])\n")
sh(f"{{sys.executable}} -m pip install -q {{_d}}")
sh(f'{{sys.executable}} -m pip install -q "numpy<2"')
sh(f'{{sys.executable}} -m pip install -q "git+https://github.com/google-deepmind/gemma.git"')

# Weightless verification: no weights downloaded, catches a broken install
# before we spend 50 GB of download and TPU time.
_r = subprocess.run([sys.executable, "-c",
    "import jax;print('jax',jax.__version__,jax.default_backend(),"
    "len(jax.devices()),jax.devices()[0].device_kind);"
    "from gemma import gm,diffusion;"
    "print('diff ckpt:',diffusion.CheckpointPath.DIFFUSIONGEMMA_26B_A4B_IT.value);"
    "print('ar   ckpt:',gm.ckpts.CheckpointPath.GEMMA4_26B_A4B_IT.value)"],
    capture_output=True, text=True)
print("VERIFY:", _r.stdout.strip() or _r.stderr.strip()[-1500:], flush=True)
if "diff ckpt" not in _r.stdout:
    raise SystemExit("install verification FAILED - do not proceed to weights")
{devices_cell}

In [ ]:
# --- generate ------------------------------------------------------------
# One model resident at a time. Weights stream from public gs://gemma-data/.
import os, sys, pathlib
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
sys.path.insert(0, "/kaggle/working/src")
!python /kaggle/working/src/generate.py --model ar --out /kaggle/working/out/ar.jsonl


In [ ]:
# --- package output ------------------------------------------------------
# Only the JSONL comes back; the ~50 GB of weights stay on the TPU.
import pathlib, json, shutil
src = pathlib.Path("/kaggle/working/out/ar.jsonl")
dst = pathlib.Path("/kaggle/working/ar.jsonl")
if src.exists():
    shutil.copy(src, dst)
    rows = [json.loads(l) for l in src.read_text().splitlines() if l.strip()]
    ok = [r for r in rows if r.get("response")]
    print(f"rows={len(rows)}  non_empty={len(ok)}  "
          f"mean_tokens={sum(r['metrics'].get('n_tokens',0) for r in ok)/max(len(ok),1):.0f}")
else:
    print("NO OUTPUT FILE")
print("done")
